# Cassava Leaf Disease Classification

| Label | Disease |
|-------|----------|
| 0 | Cassava Bacterial Blight (CBB) |
| 1 | Cassava Brown Streak Disease (CBSD) |
| 2 | Cassava Green Mottle (CGM) |
| 3 | Cassava Mosaic Disease (CMD) |
| 4 | Healthy |

## Summary

### Changes vs V2

| Component | V2 | V3 (This Notebook) |
|-----------|----|-----------------|
| **Loss** | CrossEntropyLoss (label_smoothing=0.1) | **BiTemperedLogisticLoss** (t1=0.5, t2=1.5, smoothing=0.1) |
| **Epochs** | 12 | 15 |
| **Augmentation** | RandomResizedCrop, HFlip, VFlip, Rotate(45°), ColorJitter, Noise/Blur | + **GridDistortion** (p=0.2), **CoarseDropout** (p=0.3) |
| **CutMix** | No | **Yes** (alpha=1.0, p=0.5) |
| **MixUp** | No | **Yes** (alpha=0.4, p=0.3) |
| **Test-Time Aug** | None | **8× TTA** |
| **CV Training** | Fold 0 only | Fold 0 only |

### Motivation

- **Bi-Tempered Logistic Loss**: standard cross-entropy has unbounded loss for confidently wrong predictions. `t1 < 1` makes the loss *bounded* for outliers (handles label noise); `t2 > 1` makes the tail of the probability distribution *heavier* (prevents overconfident softmax). This is especially relevant for cassava where inter-class visual similarity causes frequent annotation errors.
- **GridDistortion**: simulates the geometric lens distortion common in field-captured leaf photos.
- **CoarseDropout**: randomly occludes image patches, preventing the model from relying on single discriminative regions.
- **CutMix + MixUp**: both create virtual training examples that act as strong regularisers, reducing overfitting on the imbalanced class distribution.
- **8× TTA**: at inference, eight geometric variants of each test image are averaged, reducing prediction variance and recovering accuracy lost to augmentation randomness.

### Key Hyperparameters

- **Model**: `tf_efficientnet_b4_ns` (unchanged)
- **Image size**: 512 × 512 (unchanged)
- **Batch size**: 16
- **Epochs**: 15
- **Optimizer**: AdamW (lr=3e-4, weight_decay=1e-5) — unchanged
- **Scheduler**: CosineAnnealingWarmRestarts (T_0=5×steps) — unchanged
- **Loss**: BiTemperedLogisticLoss (t1=0.5, t2=1.5, label_smoothing=0.1)
- **CutMix**: alpha=1.0, p=0.5
- **MixUp**: alpha=0.4, p=0.3
- **TTA**: 8 variants
- **CV setup**: StratifiedKFold (5 folds; trained on fold 0)

In [ ]:
# Install / upgrade dependencies
!pip install -q timm "albumentations==1.3.1" opencv-python-headless

In [ ]:
import os
import json
import random
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from contextlib import nullcontext

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts

# AMP — prefer modern torch.amp, fall back for older torch
try:
    from torch.amp import GradScaler, autocast
    _AMP_DEVICE_ARG = True
except ImportError:
    from torch.cuda.amp import GradScaler, autocast
    _AMP_DEVICE_ARG = False

import timm
import albumentations as A
from albumentations.pytorch import ToTensorV2
from sklearn.model_selection import StratifiedKFold

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
class CFG:
    # Paths (Kaggle)
    data_dir      = '/kaggle/input/cassava-leaf-disease-classification'
    train_csv     = data_dir + '/train.csv'
    train_images  = data_dir + '/train_images'
    test_images   = data_dir + '/test_images'
    label_map_path = data_dir + '/label_num_to_disease_map.json'
    sample_submission = data_dir + '/sample_submission.csv'
    output_dir    = '/kaggle/working'

    # Model
    model_name    = 'tf_efficientnet_b4_ns'
    pretrained    = True
    num_classes   = 5
    model_save_prefix = 'best_model_v3'

    # Training
    seed          = 42
    num_folds     = 5
    fold          = 0
    num_epochs    = 15
    batch_size    = 16
    num_workers   = 2
    image_size    = 512

    # Optimizer
    lr            = 3e-4
    weight_decay  = 1e-5

    # Scheduler
    T_0           = 5
    eta_min       = 1e-6

    # Loss — Bi-Tempered Logistic
    bi_tempered_t1  = 0.5
    bi_tempered_t2  = 1.5
    label_smoothing = 0.1

    # AMP
    use_amp       = torch.cuda.is_available()

    # Augmentation mixing
    cutmix_alpha  = 1.0
    mixup_alpha   = 0.4
    cutmix_prob   = 0.5
    mixup_prob    = 0.3

    # TTA
    num_tta       = 8

    # ImageNet stats
    mean = [0.485, 0.456, 0.406]
    std  = [0.229, 0.224, 0.225]


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(CFG.seed)

assert os.path.exists(CFG.data_dir),  f'data_dir not found: {CFG.data_dir}'
assert os.path.exists(CFG.train_csv), f'train.csv not found: {CFG.train_csv}'
print('Config loaded.')
print(f'  model       : {CFG.model_name}')
print(f'  image_size  : {CFG.image_size}')
print(f'  use_amp     : {CFG.use_amp}')
print(f'  loss        : BiTemperedLogistic (t1={CFG.bi_tempered_t1}, t2={CFG.bi_tempered_t2})')

## Bi-Tempered Logistic Loss

Standard cross-entropy loss suffers from unbounded gradients for outliers with high confidence. The bi-tempered variant introduces two temperature parameters:
- **t1 < 1** (here 0.5): replaces the log with a bounded `log_t`, making the loss robust to noisy labels by capping the contribution of confidently mislabelled samples.
- **t2 > 1** (here 1.5): replaces softmax with a heavy-tailed `exp_t`, preventing over-peaked probability distributions and improving calibration.

Reference: *Amid et al. (2019) — Robust Bi-Tempered Logistic Loss Based on Bregman Divergences*

In [ ]:
def log_t(u, t):
    """Compute log_t (tempered logarithm)."""
    if t == 1.0:
        return torch.log(u)
    return (u.pow(1.0 - t) - 1.0) / (1.0 - t)


def exp_t(u, t):
    """Compute exp_t (tempered exponential)."""
    if t == 1.0:
        return torch.exp(u)
    return torch.relu(1.0 + (1.0 - t) * u).pow(1.0 / (1.0 - t))


def compute_normalization(activations, t, num_iters=5):
    """Compute the tempered softmax normalisation constant."""
    mu = activations.max(dim=-1, keepdim=True).values
    effective_dim = torch.tensor(
        activations.shape[-1], dtype=activations.dtype, device=activations.device
    )
    z = mu + ((effective_dim * (1 - t)).pow(1.0 / (2.0 - t))) / (2.0 - t)
    for _ in range(num_iters):
        z = mu + (exp_t(activations - z, t)).sum(dim=-1, keepdim=True)
    return z


def tempered_softmax(activations, t):
    """Tempered softmax function."""
    if t == 1.0:
        return F.softmax(activations, dim=-1)
    normalization = compute_normalization(activations, t)
    return exp_t(activations - normalization, t)


class BiTemperedLogisticLoss(nn.Module):
    """
    Bi-Tempered Logistic Loss.
      t1 < 1 : bounded loss — robust to noisy labels (outlier samples clipped)
      t2 > 1 : heavy-tailed distribution — prevents overconfident predictions
    """
    def __init__(self, t1=0.5, t2=1.5, label_smoothing=0.1, reduction='mean'):
        super().__init__()
        self.t1 = t1
        self.t2 = t2
        self.label_smoothing = label_smoothing
        self.reduction = reduction

    def forward(self, logits, labels):
        """
        logits : (B, C) raw model outputs
        labels : (B,) integer class indices OR (B, C) soft label tensor (from CutMix/MixUp)
        """
        num_classes = logits.shape[-1]

        if labels.dim() == 1:
            labels_onehot = F.one_hot(labels.long(), num_classes).float()
        else:
            labels_onehot = labels.float()

        if self.label_smoothing > 0:
            labels_onehot = (
                labels_onehot * (1 - self.label_smoothing)
                + self.label_smoothing / num_classes
            )

        probabilities = tempered_softmax(logits, self.t2)

        loss_values = (
            labels_onehot * log_t(labels_onehot + 1e-10, self.t1)
            - labels_onehot * log_t(probabilities, self.t1)
            - (1.0 / (2.0 - self.t1)) * (
                labels_onehot.pow(2.0 - self.t1) - probabilities.pow(2.0 - self.t1)
            )
        ).sum(dim=-1)

        if self.reduction == 'mean':
            return loss_values.mean()
        elif self.reduction == 'sum':
            return loss_values.sum()
        return loss_values


print('BiTemperedLogisticLoss defined.')

In [ ]:
# Load training CSV and label map
train_df = pd.read_csv(CFG.train_csv)
print(f'Train samples: {len(train_df)}')
print(train_df.head())

with open(CFG.label_map_path, 'r') as f:
    label_map = {int(k): v for k, v in json.load(f).items()}

print('\nLabel map:')
for k, v in label_map.items():
    print(f'  {k}: {v}')

class_counts = train_df['label'].value_counts().sort_index()
class_names  = [label_map[i] for i in class_counts.index]

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(class_names, class_counts.values,
              color=['#e74c3c', '#e67e22', '#f1c40f', '#2ecc71', '#3498db'])
ax.set_title('Class Distribution in Training Set', fontsize=14, fontweight='bold')
ax.set_xlabel('Disease Class'); ax.set_ylabel('Number of Images')
ax.tick_params(axis='x', rotation=20)
for bar, count in zip(bars, class_counts.values):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 30,
            str(count), ha='center', va='bottom', fontweight='bold')
plt.tight_layout(); plt.show()

print('\nClass counts:')
for i, (name, count) in enumerate(zip(class_names, class_counts.values)):
    print(f'  [{i}] {name}: {count} ({100*count/len(train_df):.1f}%)')

In [ ]:
# Augmentation transforms
# New vs V2: GridDistortion (p=0.2) and CoarseDropout (p=0.3) added to training pipeline

def get_train_transforms():
    return A.Compose([
        A.RandomResizedCrop(CFG.image_size, CFG.image_size, scale=(0.7, 1.0), p=1.0),
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.5),
        A.Rotate(limit=45, p=0.5),
        A.ColorJitter(
            brightness=0.3, contrast=0.3,
            saturation=0.3, hue=0.1, p=0.5
        ),
        A.OneOf([
            A.GaussNoise(var_limit=(10.0, 50.0), p=1.0),
            A.GaussianBlur(blur_limit=(3, 7), p=1.0),
            A.MotionBlur(blur_limit=7, p=1.0),
        ], p=0.3),
        A.GridDistortion(num_steps=5, distort_limit=0.3, p=0.2),
        A.CoarseDropout(
            max_holes=8,
            max_height=CFG.image_size // 16,
            max_width=CFG.image_size // 16,
            min_holes=1,
            fill_value=0,
            p=0.3
        ),
        A.Normalize(mean=CFG.mean, std=CFG.std),
        ToTensorV2(),
    ])

def get_val_transforms():
    return A.Compose([
        A.Resize(CFG.image_size, CFG.image_size),
        A.Normalize(mean=CFG.mean, std=CFG.std),
        ToTensorV2(),
    ])

def get_tta_transforms():
    """8 geometric TTA variants: original, h-flip, v-flip, both flips, rotate,
    rotate+h-flip, crop, crop+h-flip."""
    _norm   = [A.Normalize(mean=CFG.mean, std=CFG.std), ToTensorV2()]
    _resize = A.Resize(CFG.image_size, CFG.image_size)
    _crop   = A.RandomResizedCrop(CFG.image_size, CFG.image_size, scale=(0.85, 1.0), p=1.0)
    return [
        A.Compose([_resize,                                                         *_norm]),
        A.Compose([_resize, A.HorizontalFlip(p=1.0),                               *_norm]),
        A.Compose([_resize, A.VerticalFlip(p=1.0),                                 *_norm]),
        A.Compose([_resize, A.HorizontalFlip(p=1.0), A.VerticalFlip(p=1.0),        *_norm]),
        A.Compose([_resize, A.Rotate(limit=90, p=1.0),                             *_norm]),
        A.Compose([_resize, A.Rotate(limit=90, p=1.0), A.HorizontalFlip(p=1.0),    *_norm]),
        A.Compose([_crop,                                                           *_norm]),
        A.Compose([_crop,   A.HorizontalFlip(p=1.0),                               *_norm]),
    ]

print('Transforms defined.')
print('  Train : RandomResizedCrop, HFlip, VFlip, Rotate(45°), ColorJitter, Noise/Blur/Motion, GridDistortion, CoarseDropout')
print('  Val   : Resize + Normalize')
print(f'  TTA   : {CFG.num_tta} variants')

In [ ]:
class CassavaDataset(Dataset):
    """Custom PyTorch Dataset for Cassava Leaf Disease images."""

    def __init__(self, df, img_dir, transform=None, is_test=False):
        self.df = df.reset_index(drop=True)
        self.img_dir = img_dir
        self.transform = transform
        self.is_test = is_test

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = os.path.join(self.img_dir, row['image_id'])

        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        if self.transform:
            image = self.transform(image=image)['image']

        if self.is_test:
            return image

        label = torch.tensor(row['label'], dtype=torch.long)
        return image, label


class TTADataset(Dataset):
    """Dataset that applies a fixed TTA transform at inference time."""

    def __init__(self, df, img_dir, transform):
        self.df = df.reset_index(drop=True)
        self.img_dir = img_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_id    = self.df.iloc[idx]['image_id']
        img_path  = os.path.join(self.img_dir, img_id)
        image     = cv2.imread(img_path)
        image     = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        image     = self.transform(image=image)['image']
        return image


print('CassavaDataset and TTADataset defined.')

In [ ]:
# StratifiedKFold split
skf = StratifiedKFold(n_splits=CFG.num_folds, shuffle=True, random_state=CFG.seed)

train_df['fold'] = -1
for fold_idx, (_, val_idx) in enumerate(skf.split(train_df, train_df['label'])):
    train_df.loc[val_idx, 'fold'] = fold_idx

print('Fold distribution:')
for f in range(CFG.num_folds):
    n = (train_df['fold'] == f).sum()
    print(f'  Fold {f}: {n} samples')

fold_train_df = train_df[train_df['fold'] != CFG.fold].reset_index(drop=True)
fold_val_df   = train_df[train_df['fold'] == CFG.fold].reset_index(drop=True)
print(f'\nUsing fold {CFG.fold}:')
print(f'  Train: {len(fold_train_df)} | Val: {len(fold_val_df)}')

train_dataset = CassavaDataset(fold_train_df, CFG.train_images, transform=get_train_transforms())
val_dataset   = CassavaDataset(fold_val_df,   CFG.train_images, transform=get_val_transforms())

train_loader = DataLoader(
    train_dataset, batch_size=CFG.batch_size, shuffle=True,
    num_workers=CFG.num_workers, pin_memory=True, drop_last=True,
)
val_loader = DataLoader(
    val_dataset, batch_size=CFG.batch_size, shuffle=False,
    num_workers=CFG.num_workers, pin_memory=True,
)

print(f'  Train batches: {len(train_loader)} | Val batches: {len(val_loader)}')

## CutMix & MixUp

Both techniques create virtual training examples that interpolate between pairs of images:

- **CutMix**: pastes a random rectangular patch from one image onto another. The label is a weighted combination based on the patch area. Preserves local texture context.
- **MixUp**: pixel-level convex combination of two images and their labels using a Beta-distributed mixing coefficient. Encourages smooth decision boundaries.

In each training step, one is applied with probability `cutmix_prob` or `mixup_prob` (mutually exclusive via cumulative check). The loss is computed as `lam * L(pred, label_a) + (1-lam) * L(pred, label_b)`. The BiTempered loss supports this natively via its soft-label input path.

In [ ]:
def rand_bbox(size, lam):
    """Compute a random bounding box for CutMix."""
    W = size[2]
    H = size[3]
    cut_rat = np.sqrt(1.0 - lam)
    cut_w = int(W * cut_rat)
    cut_h = int(H * cut_rat)
    cx = np.random.randint(W)
    cy = np.random.randint(H)
    x1 = np.clip(cx - cut_w // 2, 0, W)
    y1 = np.clip(cy - cut_h // 2, 0, H)
    x2 = np.clip(cx + cut_w // 2, 0, W)
    y2 = np.clip(cy + cut_h // 2, 0, H)
    return x1, y1, x2, y2


def cutmix(images, labels, alpha=1.0):
    """Apply CutMix. Returns mixed images and label pair (a, b, lam)."""
    lam = np.random.beta(alpha, alpha)
    rand_index = torch.randperm(images.size(0), device=images.device)
    labels_a, labels_b = labels, labels[rand_index]
    x1, y1, x2, y2 = rand_bbox(images.size(), lam)
    images[:, :, x1:x2, y1:y2] = images[rand_index, :, x1:x2, y1:y2]
    lam = 1 - ((x2 - x1) * (y2 - y1)) / (images.size(-1) * images.size(-2))
    return images, labels_a, labels_b, lam


def mixup(images, labels, alpha=0.4):
    """Apply MixUp. Returns mixed images and label pair (a, b, lam)."""
    lam = np.random.beta(alpha, alpha)
    rand_index = torch.randperm(images.size(0), device=images.device)
    labels_a, labels_b = labels, labels[rand_index]
    mixed_images = lam * images + (1 - lam) * images[rand_index]
    return mixed_images, labels_a, labels_b, lam


def mixed_criterion(criterion, pred, y_a, y_b, lam):
    """Weighted combination of losses for CutMix/MixUp pairs."""
    return lam * criterion(pred, y_a) + (1 - lam) * criterion(pred, y_b)


print('CutMix, MixUp and mixed_criterion defined.')

In [ ]:
def build_model(pretrained=CFG.pretrained):
    model = timm.create_model(CFG.model_name, pretrained=pretrained)
    n_features = model.classifier.in_features
    model.classifier = nn.Linear(n_features, CFG.num_classes)
    return model.to(device)


model = build_model()

total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Model: {CFG.model_name}')
print(f'  Total parameters    : {total_params:,}')
print(f'  Trainable parameters: {trainable_params:,}')

In [ ]:
# Loss, AMP, Optimizer, Scheduler
criterion = BiTemperedLogisticLoss(
    t1=CFG.bi_tempered_t1,
    t2=CFG.bi_tempered_t2,
    label_smoothing=CFG.label_smoothing
)
print(f'Loss: BiTemperedLogisticLoss(t1={CFG.bi_tempered_t1}, t2={CFG.bi_tempered_t2}, smoothing={CFG.label_smoothing})')

scaler = GradScaler() if CFG.use_amp else None
print(f'AMP : {CFG.use_amp}')


def _autocast_ctx():
    if not CFG.use_amp or device.type != 'cuda':
        return nullcontext()
    if _AMP_DEVICE_ARG:
        return autocast(device_type='cuda')
    return autocast()


optimizer = optim.AdamW(model.parameters(), lr=CFG.lr, weight_decay=CFG.weight_decay)
scheduler = CosineAnnealingWarmRestarts(
    optimizer,
    T_0=CFG.T_0 * len(train_loader),
    eta_min=CFG.eta_min
)

print(f'Optimizer : AdamW (lr={CFG.lr}, weight_decay={CFG.weight_decay})')
print(f'Scheduler : CosineAnnealingWarmRestarts (T_0={CFG.T_0}×{len(train_loader)}, eta_min={CFG.eta_min})')

## Training

The training loop now randomly applies CutMix or MixUp to each batch before the forward pass. The loss is computed as a weighted combination across the two mixed label targets.

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, scaler, scheduler, device, epoch):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    pbar = tqdm(loader, desc=f'Epoch {epoch+1} [Train]', leave=False)
    for images, labels in pbar:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        # Randomly apply CutMix or MixUp
        r = random.random()
        use_mix = False
        if r < CFG.cutmix_prob:
            images, labels_a, labels_b, lam = cutmix(images, labels, alpha=CFG.cutmix_alpha)
            use_mix = True
        elif r < CFG.cutmix_prob + CFG.mixup_prob:
            images, labels_a, labels_b, lam = mixup(images, labels, alpha=CFG.mixup_alpha)
            use_mix = True

        optimizer.zero_grad(set_to_none=True)

        with _autocast_ctx():
            outputs = model(images)
            if use_mix:
                loss = mixed_criterion(criterion, outputs, labels_a, labels_b, lam)
            else:
                loss = criterion(outputs, labels)

        if scaler is not None:
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            loss.backward()
            optimizer.step()

        scheduler.step()

        preds = outputs.argmax(dim=1)
        orig_labels = labels_a if use_mix else labels
        correct += (preds == orig_labels).sum().item()
        running_loss += loss.item() * images.size(0)
        total += images.size(0)

        pbar.set_postfix({'loss': f'{loss.item():.4f}', 'acc': f'{correct/total:.4f}'})

    return running_loss / total, correct / total


print('train_one_epoch() defined.')

In [ ]:
def validate_one_epoch(model, loader, criterion, device, epoch):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0

    pbar = tqdm(loader, desc=f'Epoch {epoch+1} [Val]  ', leave=False)
    with torch.no_grad():
        for images, labels in pbar:
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            with _autocast_ctx():
                outputs = model(images)
                loss = criterion(outputs, labels)

            running_loss += loss.item() * images.size(0)
            preds = outputs.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total   += labels.size(0)

            pbar.set_postfix({'loss': f'{loss.item():.4f}', 'acc': f'{correct/total:.4f}'})

    return running_loss / total, correct / total


print('validate_one_epoch() defined.')

In [ ]:
history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}

best_val_acc = 0.0
best_model_path = os.path.join(CFG.output_dir, f'{CFG.model_save_prefix}_fold{CFG.fold}.pth')

print(f'Starting training for {CFG.num_epochs} epochs...')
print(f'{"Epoch":>5} | {"Train Loss":>10} | {"Train Acc":>9} | {"Val Loss":>8} | {"Val Acc":>7} | {"LR":>10}')
print('-' * 65)

for epoch in range(CFG.num_epochs):
    train_loss, train_acc = train_one_epoch(
        model, train_loader, criterion, optimizer, scaler, scheduler, device, epoch
    )
    val_loss, val_acc = validate_one_epoch(model, val_loader, criterion, device, epoch)

    current_lr = optimizer.param_groups[0]['lr']

    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)

    print(f'{epoch+1:>5} | {train_loss:>10.4f} | {train_acc:>9.4f} | {val_loss:>8.4f} | {val_acc:>7.4f} | {current_lr:>10.2e}')

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), best_model_path)
        print(f'         --> New best model saved! Val Acc: {best_val_acc:.4f}')

print(f'\nTraining complete. Best Val Acc: {best_val_acc:.4f}')

In [ ]:
# Plot training curves
epochs_range = range(1, CFG.num_epochs + 1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(epochs_range, history['train_loss'], 'b-o', label='Train Loss', linewidth=2)
ax1.plot(epochs_range, history['val_loss'],   'r-o', label='Val Loss',   linewidth=2)
ax1.set_title('Training & Validation Loss', fontsize=13, fontweight='bold')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss')
ax1.legend(); ax1.grid(True, alpha=0.3)

ax2.plot(epochs_range, history['train_acc'], 'b-o', label='Train Acc', linewidth=2)
ax2.plot(epochs_range, history['val_acc'],   'r-o', label='Val Acc',   linewidth=2)
ax2.axhline(y=best_val_acc, color='green', linestyle='--', alpha=0.7,
            label=f'Best Val Acc: {best_val_acc:.4f}')
ax2.set_title('Training & Validation Accuracy', fontsize=13, fontweight='bold')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Accuracy')
ax2.legend(); ax2.grid(True, alpha=0.3)

plt.suptitle(f'{CFG.model_name} — Fold {CFG.fold} Training Curves',
             fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

## Inference with 8× Test-Time Augmentation

At inference time, each test image is processed through 8 geometric transforms. The softmax probabilities are averaged across all 8 variants before taking the argmax. This reduces prediction variance, especially for borderline cases where a single crop/orientation might be misclassified.

Note: The next version (V4) extends this with 5-fold ensemble averaging on top of TTA.

In [ ]:
def predict_with_tta(model, df, img_dir):
    """Run all TTA transforms and return averaged softmax probabilities."""
    model.eval()
    tta_transforms = get_tta_transforms()
    all_probs = []

    for i, tta_t in enumerate(tta_transforms):
        dataset = TTADataset(df, img_dir, tta_t)
        loader  = DataLoader(
            dataset, batch_size=CFG.batch_size, shuffle=False,
            num_workers=CFG.num_workers, pin_memory=True
        )
        batch_probs = []
        with torch.no_grad():
            for images in tqdm(loader, desc=f'TTA {i+1}/{len(tta_transforms)}', leave=False):
                images = images.to(device, non_blocking=True)
                with _autocast_ctx():
                    logits = model(images)
                probs = F.softmax(logits, dim=1)
                batch_probs.append(probs.cpu().numpy())
        all_probs.append(np.concatenate(batch_probs, axis=0))

    return np.mean(all_probs, axis=0)   # (N, 5) averaged across TTA variants


print('predict_with_tta() defined.')

In [ ]:
# Load best checkpoint
inference_model = build_model(pretrained=False)
inference_model.load_state_dict(torch.load(best_model_path, map_location=device))
inference_model.to(device)
print(f'Loaded best model from: {best_model_path}')
print(f'Best validation accuracy: {best_val_acc:.4f}')

# Build test dataframe
test_df = pd.read_csv(CFG.sample_submission)
print(f'Test samples: {len(test_df)}')

# Run TTA inference
print(f'Running {CFG.num_tta}× TTA inference...')
probs = predict_with_tta(inference_model, test_df, CFG.test_images)
all_preds = np.argmax(probs, axis=1)

print(f'Inference complete. Predictions: {len(all_preds)}')
unique, counts = np.unique(all_preds, return_counts=True)
print('Prediction distribution:')
for cls, cnt in zip(unique, counts):
    print(f'  [{cls}] {label_map[cls]}: {cnt}')

In [ ]:
submission_df = pd.DataFrame({'image_id': test_df['image_id'], 'label': all_preds})
submission_path = os.path.join(CFG.output_dir, 'submission.csv')
submission_df.to_csv(submission_path, index=False)

print(f'Submission saved to: {submission_path}')
print(f'Shape: {submission_df.shape}')
print(submission_df.head(10))

assert len(submission_df) == len(test_df), 'Row count mismatch!'
assert submission_df['label'].between(0, 4).all(), 'Invalid label values!'
print('\nSanity checks passed. Ready to submit!')